sam2+unetpp (cross domain)


In [ ]:
# --- Paired Statistical Comparison Between Two Fusion Methods ---
import pandas as pd
from scipy.stats import wilcoxon, ttest_rel

# ========== USER CONFIG ==========
csv_1 = "/content/drive/MyDrive/Thesis-ontariotech/Dataset/ACDC_nnunet/acdc_fused_sam2_unetpp/Final_dst_fusion_metrics.csv"     # e.g. DST fusion results
csv_2 = "/content/drive/MyDrive/Thesis-ontariotech/ablations/simple_averaging/sam2_unetpp_crossdomain.csv" #simple averaging results
method_1 = "DST Fusion"
method_2 = "Simple Averaging"
output_csv = "/content/statistical_comparison_results_allmetrics.csv"
# =================================

# --- Load CSVs ---
df1 = pd.read_csv(csv_1)
df2 = pd.read_csv(csv_2)

# Filter out summary rows
df1 = df1[~df1["Case"].isin(["Average", "Std", "Var", "Mean ± Std"])]
df2 = df2[~df2["Case"].isin(["Average", "Std", "Var", "Mean ± Std"])]

# Align cases
common_cases = sorted(set(df1["Case"]).intersection(set(df2["Case"])))
df1 = df1[df1["Case"].isin(common_cases)].sort_values("Case")
df2 = df2[df2["Case"].isin(common_cases)].sort_values("Case")
print(f"Common evaluation cases: {len(common_cases)}")

# --- Metric columns to compare (all Dice & IoU, per class + macro) ---
metric_cols = [
    "fused_Dice_LV", "fused_Dice_RV", "fused_Dice_Myo",
    "fused_IoU_LV", "fused_IoU_RV", "fused_IoU_Myo",
    "Macro_Dice_Fused", "Macro_IoU_Fused"
]

results = []
for metric in metric_cols:
    vals1 = df1[metric].astype(float).values
    vals2 = df2[metric].astype(float).values

    # Wilcoxon signed-rank (non-parametric, paired)
    stat_w, p_w = wilcoxon(vals1, vals2, alternative="greater")
    # Paired t-test (optional, parametric)
    stat_t, p_t = ttest_rel(vals1, vals2)

    mean1, mean2 = vals1.mean(), vals2.mean()
    diff_mean = mean1 - mean2

    results.append({
        "Metric": metric,
        f"Mean {method_1}": mean1,
        f"Mean {method_2}": mean2,
        "Mean Diff": diff_mean,
        "Wilcoxon Stat": stat_w,
        "Wilcoxon p-value": p_w,
        "Paired t Stat": stat_t,
        "Paired t p-value": p_t,
    })

# --- Save & Print ---
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print("\n=== Statistical Comparison Summary ===")
print(results_df.round(5))
print(f"\nSaved full table to: {output_csv}")

# --- Thesis Summary Report ---
print("\n=== Thesis Summary (for text inclusion) ===")
for _, r in results_df.iterrows():
    p = r["Wilcoxon p-value"]
    sig = "significant (p < 0.05)" if p < 0.05 else "not significant"
    print(f"{r['Metric']}: {method_1} vs {method_2} → p = {p:.4e} ({sig})")


Common evaluation cases: 80

=== Statistical Comparison Summary ===
             Metric  Mean DST Fusion  Mean Simple Averaging  Mean Diff  \
0     fused_Dice_LV          0.85100                0.76805    0.08295   
1     fused_Dice_RV          0.84557                0.59931    0.24626   
2    fused_Dice_Myo          0.74220                0.65532    0.08689   
3      fused_IoU_LV          0.74818                0.64521    0.10297   
4      fused_IoU_RV          0.74251                0.45675    0.28576   
5     fused_IoU_Myo          0.60023                0.51445    0.08577   
6  Macro_Dice_Fused          0.81292                0.67423    0.13870   
7   Macro_IoU_Fused          0.69697                0.53881    0.15817   

   Wilcoxon Stat  Wilcoxon p-value  Paired t Stat  Paired t p-value  
0         2760.0           0.00000        5.87542           0.00000  
1         3088.0           0.00000        9.86619           0.00000  
2         2542.0           0.00000        4.50862      

extra

In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/Thesis-ontariotech/Dataset/mnmchallengefinetune/fused-calib/std_var_sam2_nnunet-dst_fusion_metrics.csv"

df = pd.read_csv(CSV_PATH)

# Identify rows
avg_row = df[df["Case"] == "Average"]
std_row = df[df["Case"] == "Std"]

assert len(avg_row) == 1 and len(std_row) == 1, "Average/Std row missing"

mean_std = {}
for col in df.columns:
    if col == "Case":
        mean_std[col] = "Mean ± Std"
    else:
        mean = avg_row.iloc[0][col]
        std  = std_row.iloc[0][col]
        mean_std[col] = f"{mean:.3f} ± {std:.3f}"

# Append
df = pd.concat([df, pd.DataFrame([mean_std])], ignore_index=True)

# Save back (or to a new file)
out_path = CSV_PATH
df.to_csv(out_path, index=False)

print("Saved:", out_path)
print(df.tail(4))


Saved: /content/drive/MyDrive/Thesis-ontariotech/Dataset/mnmchallengefinetune/fused-calib/std_var_sam2_nnunet-dst_fusion_metrics.csv
           Case     nn_Dice_LV     nn_Dice_RV    nn_Dice_Myo      nn_IoU_LV  \
124     Average       0.939285       0.866626       0.915322        0.88753   
125         Std       0.035385        0.04594       0.064675       0.060312   
126         Var       0.007726       0.007439       0.009892       0.009039   
127  Mean ± Std  0.939 ± 0.035  0.867 ± 0.046  0.915 ± 0.065  0.888 ± 0.060   

         nn_IoU_RV     nn_IoU_Myo    nn_Dice_All     nn_IoU_All  \
124       0.767411         0.8493       0.912403       0.840443   
125       0.068488       0.092202       0.031852       0.052192   
126        0.00853       0.012982        0.00716       0.007633   
127  0.767 ± 0.068  0.849 ± 0.092  0.912 ± 0.032  0.840 ± 0.052   

       sam_Dice_LV  ...  fused_Dice_LV  fused_Dice_RV fused_Dice_Myo  \
124        0.77777  ...       0.944464       0.871427       0.9

REQUIRED

MedSAM2 + nnUNet — cross-domain

SAM2 + nnUNet — cross-domain

SAM2 + UNet++ — cross-domain

SAM2 + nnUNet — in-domain (optional but reasonable)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

# ===================== CONFIG =====================
METRIC_COL = "Macro_Dice_Fused"
SUMMARY_ROWS = ["Average", "Std", "Var", "Mean ± Std"]
N_BOOT = 10000
SEED = 42
# =================================================


def load_casewise_metric(csv_path, metric_col=METRIC_COL):
    df = pd.read_csv(csv_path)

    # remove summary rows
    df = df[~df["Case"].isin(SUMMARY_ROWS)]

    series = df.set_index("Case")[metric_col].astype(float)
    return series


def bootstrap_ci(diffs, n_boot=N_BOOT, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    diffs = np.asarray(diffs)

    boot_samples = rng.choice(diffs, size=(n_boot, len(diffs)), replace=True)
    boot_medians = np.median(boot_samples, axis=1)

    lo = np.quantile(boot_medians, alpha / 2)
    hi = np.quantile(boot_medians, 1 - alpha / 2)
    return lo, hi


def holm_bonferroni(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)

    adjusted = np.empty(m)
    for k, idx in enumerate(order):
        adjusted[idx] = min((m - k) * pvals[idx], 1.0)

    # enforce monotonicity
    for k in range(m - 2, -1, -1):
        i1 = order[k]
        i2 = order[k + 1]
        adjusted[i1] = min(adjusted[i1], adjusted[i2])

    return adjusted


def paired_test(dst_csv, avg_csv, label):
    dst = load_casewise_metric(dst_csv)
    avg = load_casewise_metric(avg_csv)

    common_cases = dst.index.intersection(avg.index)
    dst = dst.loc[common_cases].values
    avg = avg.loc[common_cases].values

    diffs = dst - avg

    # Wilcoxon signed-rank (paired)
    stat, p = wilcoxon(diffs, zero_method="wilcox", alternative="two-sided")

    median_delta = np.median(diffs)
    ci_lo, ci_hi = bootstrap_ci(diffs)

    return {
        "Pairing / Domain": label,
        "n": len(diffs),
        "ΔDice (median)": median_delta,
        "CI_low": ci_lo,
        "CI_high": ci_hi,
        "p_raw": p
    }


# =====================EXPERIMENT PAIRS=====================
experiments = [
    (
      "/content/drive/MyDrive/Thesis-ontariotech/Dataset/ACDC_nnunet/fused_medsam2_nnunet/Final_dst_fusion_metrics.csv",
      "/content/drive/MyDrive/Thesis-ontariotech/ablations/simple_averaging/medsam2_nnunet_crossdomain.csv",
      "MEDSAM2 + nnU-Net (cross-domain)"
    ),
    (
      "/content/drive/MyDrive/Thesis-ontariotech/Dataset/mnmchallengefinetune/fused-calib/std_var_sam2_nnunet-dst_fusion_metrics.csv",
      "/content/drive/MyDrive/Thesis-ontariotech/ablations/simple_averaging/sam2_nnunet_indomain.csv",
      "SAM2 + nnU-Net (in-domain)"
    ),
    (
      "/content/drive/MyDrive/Thesis-ontariotech/Dataset/ACDC_nnunet/fused_sam2_nnunet/Final_dst_fusion_metrics.csv",
      "/content/drive/MyDrive/Thesis-ontariotech/ablations/simple_averaging/sam2_nnunet_crossdomain.csv",
      "SAM2 + nnU-Net (cross-domain)"
    ),
    (
      "/content/drive/MyDrive/Thesis-ontariotech/Dataset/ACDC_nnunet/acdc_fused_sam2_unetpp/Final_dst_fusion_metrics.csv",
      "/content/drive/MyDrive/Thesis-ontariotech/ablations/simple_averaging/sam2_unetpp_crossdomain.csv",
      "SAM2 + UNet++ (cross-domain)"
    ),
]


# ===================== RUN TESTS =====================
rows = [paired_test(dst, avg, label) for dst, avg, label in experiments]

pvals = [r["p_raw"] for r in rows]
p_holm = holm_bonferroni(pvals)

for r, ph in zip(rows, p_holm):
    r["p_holm"] = ph

# ===================== FINAL TABLE =====================
df = pd.DataFrame(rows)
df["ΔDice (median)"] = df["ΔDice (median)"].round(4)
df["CI"] = df.apply(lambda r: f"[{r.CI_low:.4f}, {r.CI_high:.4f}]", axis=1)
df["p_raw"] = df["p_raw"].apply(lambda x: f"{x:.2e}")
df["p_holm"] = df["p_holm"].apply(lambda x: f"{x:.2e}")

final_cols = [
    "Pairing / Domain",
    "n",
    "ΔDice (median)",
    "CI",
    "p_raw",
    "p_holm"
]

final_df = df[final_cols]

# Save
out_csv = "statistical_significance_table.csv"
final_df.to_csv(out_csv, index=False)

print("\n=== FINAL STATISTICAL SIGNIFICANCE TABLE ===")
print(final_df)
print(f"\nSaved to: {out_csv}")



=== FINAL STATISTICAL SIGNIFICANCE TABLE ===
                   Pairing / Domain    n  ΔDice (median)                CI  \
0  MEDSAM2 + nnU-Net (cross-domain)   80          0.0113  [0.0054, 0.0190]   
1        SAM2 + nnU-Net (in-domain)  124          0.0091  [0.0060, 0.0110]   
2     SAM2 + nnU-Net (cross-domain)   80          0.0033  [0.0006, 0.0058]   
3      SAM2 + UNet++ (cross-domain)   80          0.0951  [0.0696, 0.1447]   

      p_raw    p_holm  
0  6.88e-09  1.38e-08  
1  2.24e-15  8.95e-15  
2  2.84e-02  2.84e-02  
3  4.11e-13  1.23e-12  

Saved to: statistical_significance_table.csv


In [ ]:
import os

In [ ]:
import os

# ================= PATHS (DO NOT MODIFY FILE NAMES) =================
nnunet_dir = "/content/drive/MyDrive/Thesis-ontariotech/Dataset/mnmchallengefinetune/nnUNet_dataset/nnUNet_results/inference_output2"
unetpp_dir = "/content/drive/MyDrive/Thesis-ontariotech/U-netpp/weak_unetpp_soft_npz-suboptimal"
sam2_dir   = "/content/drive/MyDrive/Thesis-ontariotech/SAM2-model/oracle_iou_main/sam2_soft_npz"

# ================= COLLECT CASE IDS =================
nnunet_cases = sorted([
    f.replace(".npz", "")
    for f in os.listdir(nnunet_dir)
    if f.endswith(".npz")
])

unetpp_cases = sorted([
    f.replace(".npz", "")
    for f in os.listdir(unetpp_dir)
    if f.endswith(".npz")
])

sam2_cases = sorted([
    f.replace("_sam2_soft.npz", "")
    for f in os.listdir(sam2_dir)
    if f.endswith("_sam2_soft.npz")
])

# ================= BASIC COUNTS =================
print("========== CASE COUNTS ==========")
print("nnUNet cases :", len(nnunet_cases))
print("UNet++ cases :", len(unetpp_cases))
print("SAM2 cases   :", len(sam2_cases))

# ================= OVERLAPS =================
common_all = sorted(set(nnunet_cases) & set(unetpp_cases) & set(sam2_cases))
print("\nCommon cases across ALL models:", len(common_all))

# ================= MISSING CASES =================
missing_in_unetpp = sorted(set(nnunet_cases) - set(unetpp_cases))
missing_in_sam2   = sorted(set(nnunet_cases) - set(sam2_cases))

print("\nMissing in UNet++:", len(missing_in_unetpp))
if missing_in_unetpp:
    print("First 20 missing in UNet++:", missing_in_unetpp[:20])

print("\nMissing in SAM2:", len(missing_in_sam2))
if missing_in_sam2:
    print("First 20 missing in SAM2:", missing_in_sam2[:20])

# ================= MODEL-SPECIFIC ONLY =================
nn_only = sorted(set(nnunet_cases) - set(unetpp_cases))
unetpp_only = sorted(set(unetpp_cases) - set(nnunet_cases))

print("\nCases ONLY in nnUNet:", len(nn_only))
if nn_only:
    print("First 20 nnUNet-only:", nn_only[:20])

print("\nCases ONLY in UNet++:", len(unetpp_only))
if unetpp_only:
    print("First 20 UNet++-only:", unetpp_only[:20])

print("\n================================")


========== CASE COUNTS ==========
nnUNet cases : 144
UNet++ cases : 144
SAM2 cases   : 144

Common cases across ALL models: 144

Missing in UNet++: 0

Missing in SAM2: 0

Cases ONLY in nnUNet: 0

Cases ONLY in UNet++: 0

